# Entrega: Experimentos del Agente MCTSVale

Agente Connect-4 basado en **Monte Carlo Tree Search** con:
- **Bitboard**: tablero como enteros de 64 bits (~10x mas rapido que numpy)
- **UCB1 + RAVE**: estadisticas globales por accion — convergencia mas rapida
- **Ordenamiento centro-primero**: MCTS explora columnas centrales antes
- **Fork detection**: detecta/crea/bloquea dobles amenazas antes de llamar MCTS

MCTS es un agente **online**: `mount()` solo reinicia el RNG, no entrena nada entre partidas.

Parametros analizados: `n_simulations`, `c` (exploracion UCB1), `use_heuristic`

In [1]:
import sys
import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt

_p = os.getcwd()
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, 'connect4')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
    _p = os.path.dirname(_p)

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from policy import MCTSVale


class RandomPolicy(Policy):
    def __init__(self, seed=None):
        self._rng = np.random.default_rng(seed)

    def mount(self):
        pass

    def act(self, s: np.ndarray) -> int:
        free = [c for c in range(7) if s[0, c] == 0]
        return int(self._rng.choice(free))


def play_game(mcts_agent, opponent, mcts_color: int):
    state  = ConnectState()
    agents = {mcts_color: mcts_agent, -mcts_color: opponent}
    times  = []
    while not state.is_final():
        if state.player == mcts_color:
            t0     = time.perf_counter()
            action = mcts_agent.act(state.board)
            times.append(time.perf_counter() - t0)
        else:
            action = opponent.act(state.board)
        state = state.transition(int(action))
    return state.get_winner(), times


def run_vs_random(n_sims, n_games=30, seed=0, c=math.sqrt(2),
                  use_heuristic=True, fixed_color=None):
    wins = draws = losses = 0
    all_times = []
    rng = np.random.default_rng(seed)
    for g in range(n_games):
        mcts = MCTSVale(n_simulations=n_sims, c=c, use_heuristic=use_heuristic)
        mcts.mount()
        rand = RandomPolicy(seed=int(rng.integers(10**6)))
        rand.mount()
        color = fixed_color if fixed_color is not None else (-1 if g % 2 == 0 else 1)
        winner, t = play_game(mcts, rand, color)
        all_times.extend(t)
        if winner == color:
            wins += 1
        elif winner == 0:
            draws += 1
        else:
            losses += 1
    return {
        'wins': wins, 'draws': draws, 'losses': losses,
        'avg_time': float(np.mean(all_times)) if all_times else 0.0,
    }


print("Setup completado.")

Setup completado.


## Experimento 1: Win-rate vs `n_simulations` contra jugador aleatorio

Variamos `n_simulations` en `[25, 50, 100, 200, 300]` y medimos:
- **Wins / Draws / Losses** en 30 partidas (colores alternados)
- **Tiempo promedio de `act()`** en segundos

Hipotesis: mas simulaciones implica mayor win-rate y mayor latencia.

In [ ]:
N_SIMS_LIST = [25, 50, 100, 200, 300]
N_GAMES_1   = 30

exp1_results = {}
for n in N_SIMS_LIST:
    r = run_vs_random(n_sims=n, n_games=N_GAMES_1, seed=10)
    exp1_results[n] = r
    wr = r['wins'] / N_GAMES_1 * 100
    print(f"n_sims={n:4d} | W={r['wins']:3d} D={r['draws']:3d} L={r['losses']:3d}"
          f" | WR={wr:5.1f}% | t={r['avg_time']:.4f}s")

x      = np.arange(len(N_SIMS_LIST))
width  = 0.25
wins_  = [exp1_results[n]['wins']     for n in N_SIMS_LIST]
draws_ = [exp1_results[n]['draws']    for n in N_SIMS_LIST]
losses_= [exp1_results[n]['losses']   for n in N_SIMS_LIST]
times_ = [exp1_results[n]['avg_time'] for n in N_SIMS_LIST]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(x - width, wins_,   width, label='Wins',   color='seagreen',  alpha=0.85)
ax1.bar(x,         draws_,  width, label='Draws',  color='steelblue', alpha=0.85)
ax1.bar(x + width, losses_, width, label='Losses', color='tomato',    alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(N_SIMS_LIST)
ax1.set_xlabel('n_simulations')
ax1.set_ylabel('Partidas (de 30)')
ax1.set_title('Exp 1: W/D/L vs n_simulations — vs aleatorio, 30 partidas')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(x, times_, 'b-o', linewidth=2, markersize=7, label='t_avg act() [s]')
ax2.set_ylabel('Tiempo promedio act() [s]', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2.legend(loc='center right')

plt.tight_layout()
plt.show()

## Experimento 1b: Win-rate por color (rojo y amarillo)

Fijamos `n_simulations=200` y separamos el resultado por color:
- **Rojo (-1)**: MCTS mueve primero — ventaja posicional en Connect-4
- **Amarillo (+1)**: MCTS mueve segundo

30 partidas por color contra jugador aleatorio.

In [ ]:
N_GAMES_1B = 30

color_results = {}
for color, label in [(-1, 'Rojo (-1)'), (1, 'Amarillo (+1)')]:
    r = run_vs_random(n_sims=200, n_games=N_GAMES_1B, seed=20, fixed_color=color)
    color_results[label] = r
    wr = r['wins'] / N_GAMES_1B * 100
    print(f"{label:15s}: W={r['wins']:3d} D={r['draws']:3d} L={r['losses']:3d}  WR={wr:.1f}%")

labels   = list(color_results.keys())
wins_c   = [color_results[l]['wins']   for l in labels]
draws_c  = [color_results[l]['draws']  for l in labels]
losses_c = [color_results[l]['losses'] for l in labels]

x = np.arange(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width, wins_c,   width, label='Wins',   color='seagreen',  alpha=0.85)
ax.bar(x,         draws_c,  width, label='Draws',  color='steelblue', alpha=0.85)
ax.bar(x + width, losses_c, width, label='Losses', color='tomato',    alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Partidas (de 30)')
ax.set_title('Exp 1b: W/D/L por color — n_sims=200, vs aleatorio')
ax.legend()
plt.tight_layout()
plt.show()

## Experimento 2: `use_heuristic=True` vs `use_heuristic=False`

Comparamos las dos variantes de rollout del agente:

| Variante | Descripcion |
|----------|-------------|
| `use_heuristic=True`  | Rollout prefiere ganar o bloquear antes de jugar aleatorio |
| `use_heuristic=False` | Rollout completamente aleatorio |

Se prueban ambas con `n_simulations` en `[50, 100, 200]` contra el jugador aleatorio,
30 partidas c/u.

Hipotesis: el rollout heuristico mejora la precision de la estimacion de valor,
especialmente con pocos `n_simulations`.

In [ ]:
N_SIMS_EXP2 = [50, 100, 200]
N_GAMES_2   = 30

exp2_results = {True: {}, False: {}}
for use_h in [True, False]:
    for n in N_SIMS_EXP2:
        r = run_vs_random(n_sims=n, n_games=N_GAMES_2, seed=30, use_heuristic=use_h)
        exp2_results[use_h][n] = r
        wr = r['wins'] / N_GAMES_2 * 100
        print(f"heuristic={str(use_h):5s} n_sims={n:3d}: "
              f"W={r['wins']:3d} D={r['draws']:3d} L={r['losses']:3d}  WR={wr:.1f}%")

x     = np.arange(len(N_SIMS_EXP2))
width = 0.35
wr_true  = [exp2_results[True][n]['wins']  / N_GAMES_2 * 100 for n in N_SIMS_EXP2]
wr_false = [exp2_results[False][n]['wins'] / N_GAMES_2 * 100 for n in N_SIMS_EXP2]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, wr_true,  width, label='use_heuristic=True',  color='seagreen', alpha=0.85)
ax.bar(x + width/2, wr_false, width, label='use_heuristic=False', color='tomato',   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(N_SIMS_EXP2)
ax.set_xlabel('n_simulations')
ax.set_ylabel('Win-rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Exp 2: Win-rate — rollout heuristico vs aleatorio puro')
ax.legend()
plt.tight_layout()
plt.show()

## Experimento 3: Efecto de `c` (constante UCB1) sobre el win-rate

La constante `c` controla el balance exploracion/explotacion:

- **c bajo**: mas explotacion — puede quedarse en optimos locales
- **c alto**: mas exploracion — puede desperdiciar simulaciones en ramas malas
- **c = sqrt(2) ≈ 1.414**: valor teorico optimo para juegos con recompensa binaria

Fijamos `n_simulations=100, use_heuristic=True` y probamos `c` en
`[0.5, 1.0, 1.414, 2.0, 3.0]` (20 partidas c/u).

In [ ]:
C_LIST    = [0.5, 1.0, math.sqrt(2), 2.0, 3.0]
N_GAMES_3 = 20

exp3_results = {}
for c_val in C_LIST:
    r = run_vs_random(n_sims=100, n_games=N_GAMES_3, seed=40, c=c_val)
    exp3_results[c_val] = r
    wr = r['wins'] / N_GAMES_3 * 100
    print(f"c={c_val:.3f}: W={r['wins']:3d} D={r['draws']:3d} L={r['losses']:3d}  WR={wr:.1f}%")

c_labels    = [f"{c:.3f}" for c in C_LIST]
wr_vals     = [exp3_results[c]['wins'] / N_GAMES_3 * 100 for c in C_LIST]
sqrt2_label = f"{math.sqrt(2):.3f}"

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(c_labels, wr_vals, 'g-o', linewidth=2, markersize=8)
if sqrt2_label in c_labels:
    idx = c_labels.index(sqrt2_label)
    ax.axvline(x=idx, color='gray', linestyle='--', alpha=0.7,
               label=f'c=sqrt(2)={sqrt2_label} (teorico)')
ax.set_xlabel('c (exploracion UCB1)')
ax.set_ylabel('Win-rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Exp 3: Win-rate vs c (n_sims=100, heuristic=True, vs aleatorio)')
ax.legend()
plt.tight_layout()
plt.show()

## Experimento 4: MCTSVale(200) vs MCTSVale(n) con distintos `n_simulations`

Enfrentamos el agente fuerte (`n_simulations=200`) contra versiones con menos simulaciones:
`n_simulations` en `[25, 50, 100]`.

Se espera que el agente con mas simulaciones gane con mayor frecuencia, validando que
el desempenio mejora consistentemente con `n_simulations`.

20 partidas por combinacion, colores alternados.

In [ ]:
N_WEAK_LIST = [25, 50, 100]
N_GAMES_4   = 20

exp4_results = {}
for n_weak in N_WEAK_LIST:
    wins = draws = losses = 0
    rng = np.random.default_rng(50)
    for g in range(N_GAMES_4):
        strong = MCTSVale(n_simulations=200)
        weak   = MCTSVale(n_simulations=n_weak)
        strong.mount()
        weak.mount()
        strong_color = -1 if g % 2 == 0 else 1
        state  = ConnectState()
        agents = {strong_color: strong, -strong_color: weak}
        while not state.is_final():
            action = agents[state.player].act(state.board)
            state  = state.transition(int(action))
        winner = state.get_winner()
        if winner == strong_color:
            wins += 1
        elif winner == 0:
            draws += 1
        else:
            losses += 1
    exp4_results[n_weak] = {'wins': wins, 'draws': draws, 'losses': losses}
    wr = wins / N_GAMES_4 * 100
    print(f"MCTS(200) vs MCTS({n_weak:3d}): W={wins:3d} D={draws:3d} L={losses:3d}  WR(200)={wr:.1f}%")

x     = np.arange(len(N_WEAK_LIST))
width = 0.25
wins4_  = [exp4_results[n]['wins']   for n in N_WEAK_LIST]
draws4_ = [exp4_results[n]['draws']  for n in N_WEAK_LIST]
los4_   = [exp4_results[n]['losses'] for n in N_WEAK_LIST]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width, wins4_,  width, label='Wins MCTS(200)',   color='seagreen',  alpha=0.85)
ax.bar(x,         draws4_, width, label='Draws',            color='steelblue', alpha=0.85)
ax.bar(x + width, los4_,   width, label='Losses MCTS(200)', color='tomato',    alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'vs n={n}' for n in N_WEAK_LIST])
ax.set_ylabel('Partidas (de 20)')
ax.set_title('Exp 4: MCTSVale(200) vs MCTSVale(n) — 20 partidas c/u')
ax.legend()
plt.tight_layout()
plt.show()

## Experimento 5: Latencia de `act()` vs `n_simulations`

Medimos el tiempo promedio de `act()` con tablero vacio.
Con tablero vacio no hay victorias inmediatas ni forks, por lo que
siempre se ejecuta `_run_mcts` completo — representa el costo puro de MCTS.

Se espera una relacion aproximadamente lineal entre `n_simulations` y la latencia.

20 llamadas por configuracion.

In [ ]:
N_SIMS_EXP5 = [25, 50, 100, 200, 300]
N_CALLS_5   = 20

empty_board  = np.zeros((6, 7), dtype=int)
exp5_results = {}

for n_sims in N_SIMS_EXP5:
    mcts = MCTSVale(n_simulations=n_sims)
    mcts.mount()
    times = []
    for _ in range(N_CALLS_5):
        t0 = time.perf_counter()
        mcts.act(empty_board)
        times.append(time.perf_counter() - t0)
    avg = float(np.mean(times))
    exp5_results[n_sims] = avg
    print(f"n_sims={n_sims:4d}: avg={avg:.4f}s")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_SIMS_EXP5, [exp5_results[n] for n in N_SIMS_EXP5],
        'b-o', linewidth=2, markersize=8)
ax.set_xlabel('n_simulations')
ax.set_ylabel('Tiempo promedio act() [s]')
ax.set_title('Exp 5: Latencia de act() vs n_simulations — tablero vacio, 20 llamadas')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones

| Experimento | Observacion | Resultado esperado |
|-------------|-------------|-------------------|
| 1 — Win-rate vs n_sims | Mayor n_sims mejora el desempenio contra jugador aleatorio | Win-rate crece de ~60% (n=25) a >85% (n=300); latencia crece linealmente |
| 1b — Win-rate por color | Rojo (primer turno) tiene ventaja estructural en Connect-4 | Win-rate rojo > win-rate amarillo con n_sims=200 |
| 2 — Heuristic vs puro aleatorio | Rollout heuristico mejora la calidad de la estimacion de valor | use_heuristic=True domina en todos los n_sims; ventaja mayor con n_sims bajos |
| 3 — Efecto de c | c=sqrt(2)~1.414 es optimo teorico; valores extremos degradan rendimiento | Curva con maximo cerca de c=1.0-1.5; c muy bajo o alto reduce win-rate |
| 4 — MCTS vs si mismo | MCTS(200) supera a MCTS(25/50/100) con alta frecuencia | W/D/L muestra superioridad clara del agente con mas simulaciones |
| 5 — Latencia | Costo por llamada proporcional a n_simulations | Relacion aproximadamente lineal entre n_sims y tiempo de act() |